# 🚀 AI Customer Support & Ticket Resolution Agent
### Generative AI + RAG + Tools + Memory + Agentic Workflow


Build a realistic TechKart customer-support agent that understands tickets, retrieves policies, checks orders, makes decisions, evaluates responses, and escalates high-risk cases.

## Final Architecture
Customer Message → LLM Analysis → Category/Priority/Sentiment → Agent → RAG + Tools + Memory → Decision → Response/Escalation → Evaluator

### Use cases
- Delivery delay
- Refund
- Cancellation
- Damaged product
- Wrong product
- Technical support
- Account issues
- Angry/high-risk customers

# STEP 1 — Install Packages

In [ ]:
!pip -q install google-genai scikit-learn pandas

# STEP 2 — Import Libraries

In [ ]:
import os, json, re
from datetime import datetime
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from google import genai
from google.colab import userdata

print("Libraries loaded.")

# STEP 3 — Configure Gemini API

In Colab open **Secrets → Add new secret** and create `GEMINI_API_KEY`.
Do not publish API keys in notebooks.

In [ ]:
API_KEY = userdata.get("GEMINI_API_KEY")
if not API_KEY:
    raise ValueError("Add GEMINI_API_KEY to Colab Secrets.")

client = genai.Client(api_key=API_KEY)

# Update this model name if your Gemini account exposes a different current model.
MODEL = "gemini-2.5-flash"

print("Gemini configured.")

# STEP 4 — Gemini Helper

In [ ]:
def ask_ai(prompt):
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )
    return response.text

print(ask_ai("Explain an AI agent in one simple sentence."))

# STEP 5 — Simulated TechKart Order Database

In [ ]:
orders = {
    "ORD1001": {"customer":"Rahul","product":"Laptop","status":"Shipped","expected_delivery":"2026-09-28","amount":65000},
    "ORD1002": {"customer":"Priya","product":"Headphones","status":"Delivered","expected_delivery":"2026-09-25","amount":3500},
    "ORD1003": {"customer":"Arjun","product":"Smartphone","status":"Processing","expected_delivery":"2026-10-02","amount":28000},
    "ORD1004": {"customer":"Sneha","product":"Laptop","status":"Delivered","expected_delivery":"2026-09-27","amount":72000}
}
display(pd.DataFrame(orders).T)

# STEP 6 — Tool: Order Lookup

In [ ]:
def get_order_status(order_id):
    order_id = order_id.upper().strip()
    return orders.get(order_id, {"error": f"Order {order_id} was not found."})

print(get_order_status("ORD1001"))

# STEP 7 — Company Policy Knowledge Base

In [ ]:
policies = {
"delivery": 'TechKart Delivery Policy:\nStandard delivery normally takes 3 to 5 business days.\nIf an order is delayed beyond its expected delivery date, verify status.\nIf delayed by more than 2 business days, escalate for investigation.',
"refund": 'TechKart Refund Policy:\nEligible customers can request refunds according to product terms.\nDo not promise a refund before eligibility is verified.\nHigh-value refund disputes require human review.',
"cancellation": 'TechKart Cancellation Policy:\nCancellation may be possible before shipment.\nAfter shipment, cancellation may not be possible.\nVerify order status before confirming cancellation.',
"damaged": 'TechKart Damaged Product Policy:\nDamaged-product cases may require photographs or other evidence.\nReplacement or investigation follows company procedure.\nDo not promise replacement before verification.',
"warranty": 'TechKart Warranty Policy:\nEligible products receive warranty support according to product terms.\nTechnical issues should be diagnosed before promising repair or replacement.'
}
print(list(policies))

# STEP 8 — Simple RAG with TF-IDF

In [ ]:
policy_names = list(policies.keys())
policy_texts = list(policies.values())
vectorizer = TfidfVectorizer()
policy_vectors = vectorizer.fit_transform(policy_texts)

def retrieve_policy(query, top_k=2):
    qv = vectorizer.transform([query])
    scores = cosine_similarity(qv, policy_vectors)[0]
    ranked = scores.argsort()[::-1][:top_k]
    return [{"policy":policy_names[i], "score":float(scores[i]), "text":policy_texts[i]} for i in ranked]

print(retrieve_policy("My laptop delivery is late", 1)[0])

# STEP 9 — Test RAG

In [ ]:
questions = [
    "I want my money back",
    "My product arrived broken",
    "Can I cancel my order?",
    "My package is late",
    "My laptop has a technical problem"
]
for q in questions:
    r = retrieve_policy(q, 1)[0]
    print(f"{q} -> {r['policy']} ({r['score']:.3f})")

# STEP 10 — AI Ticket Classification

In [ ]:
def analyze_ticket(message):
    prompt = f'''
You are a customer support classifier for TechKart.

Return ONLY valid JSON:
{{
  "category": "Delivery|Refund|Cancellation|Damaged Product|Wrong Product|Technical Support|Account|Other",
  "priority": "Low|Medium|High|Critical",
  "sentiment": "Positive|Neutral|Frustrated|Angry",
  "summary": "...",
  "order_id": "ORD#### or null",
  "requires_human": true/false
}}

Customer message:
{message}
'''
    raw = ask_ai(prompt)
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if not match:
        return {"raw_response": raw}
    try:
        return json.loads(match.group())
    except:
        return {"raw_response": raw}

msg = "My laptop hasn't arrived. Expected delivery was yesterday. Order ORD1001."
print(json.dumps(analyze_ticket(msg), indent=2))

# STEP 11 — Challenge: Classify Multiple Tickets

In [ ]:
messages = [
    "My laptop hasn't arrived yet. Order ORD1001.",
    "I received the wrong product. Order ORD1002.",
    "I want to cancel my smartphone order ORD1003.",
    "My laptop arrived damaged. Order ORD1004.",
    "My headphones stopped working after two weeks.",
    "I am extremely angry. Nobody is helping me!"
]
for msg in messages:
    print("\nCUSTOMER:", msg)
    print(json.dumps(analyze_ticket(msg), indent=2))

# STEP 12 — Deterministic Python Tool: Extract Order ID

In [ ]:
def extract_order_id(text):
    match = re.search(r'ORD\d{4}', text.upper())
    return match.group(0) if match else None

for t in ["Check ORD1001", "My order is ORD1003", "I don't know my order"]:
    print(t, "=>", extract_order_id(t))

# STEP 13 — Create Agent Tools

In [ ]:
tickets = []

def check_policy(issue):
    return retrieve_policy(issue, 1)[0]

def create_support_ticket(customer, category, priority, message, order_id=None):
    ticket = {
        "ticket_id": f"TKT{len(tickets)+1001}",
        "customer": customer,
        "category": category,
        "priority": priority,
        "message": message,
        "order_id": order_id,
        "status": "Open",
        "created_at": datetime.now().isoformat(timespec="seconds")
    }
    tickets.append(ticket)
    return ticket

def escalate_to_human(reason, ticket):
    return {"escalated": True, "reason": reason, "ticket": ticket}

print("Agent tools ready.")

# STEP 14 — Build the Agent Decision Layer

In [ ]:
def support_agent(customer_message, customer_name="Customer"):
    analysis = analyze_ticket(customer_message)
    order_id = analysis.get("order_id") or extract_order_id(customer_message)
    order_info = get_order_status(order_id) if order_id else None
    policy = check_policy(customer_message)

    ticket = create_support_ticket(
        customer_name,
        analysis.get("category", "Other"),
        analysis.get("priority", "Medium"),
        customer_message,
        order_id
    )

    prompt = f'''
You are TechKart's support decision agent.
Use ONLY supplied information.

Customer message:
{customer_message}

Ticket analysis:
{json.dumps(analysis, indent=2)}

Order information:
{json.dumps(order_info, indent=2)}

Relevant policy:
{json.dumps(policy, indent=2)}

Return ONLY JSON:
{{
  "action": "Respond" or "Escalate",
  "reason": "...",
  "customer_response": "..."
}}

Never invent order details. Never promise a refund, replacement or cancellation
unless the supplied information supports it.
'''
    raw = ask_ai(prompt)
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    try:
        decision = json.loads(match.group()) if match else {"raw_response": raw}
    except:
        decision = {"raw_response": raw}

    return {"analysis":analysis, "order_info":order_info, "policy":policy,
            "ticket":ticket, "decision":decision}

result = support_agent(msg, "Rahul")
print(json.dumps(result, indent=2))

# STEP 15 — AI Response Evaluator

In [ ]:
def evaluate_response(customer_message, response, context):
    prompt = f'''
You are a quality evaluator for TechKart.

Customer:
{customer_message}

Proposed response:
{response}

Context:
{json.dumps(context, indent=2)}

Check relevance, factual grounding, policy compliance,
no invented information, and professional tone.

Return ONLY JSON:
{{
  "passed": true/false,
  "issues": [],
  "suggested_improvement": ""
}}
'''
    raw = ask_ai(prompt)
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if not match:
        return {"raw_response": raw}
    try:
        return json.loads(match.group())
    except:
        return {"raw_response": raw}

response = result["decision"].get("customer_response", "")
print(response)
print(json.dumps(evaluate_response(msg, response, result), indent=2))

# STEP 16 — Human-in-the-Loop

In [ ]:
def needs_human_review(analysis, order_info):
    if analysis.get("priority") in ["Critical", "High"]:
        return True
    if str(analysis.get("sentiment", "")).lower() in ["angry", "very negative"]:
        return True
    if isinstance(order_info, dict) and order_info.get("amount", 0) >= 60000:
        return True
    return False

print(needs_human_review(result["analysis"], result["order_info"]))

# STEP 17 — Conversation Memory

In [ ]:
conversation = []

def add_message(role, message):
    conversation.append({
        "role": role,
        "message": message,
        "timestamp": datetime.now().isoformat(timespec="seconds")
    })

add_message("customer", "My laptop hasn't arrived.")
add_message("assistant", "Please provide your order ID.")
add_message("customer", "ORD1001")

display(pd.DataFrame(conversation))

# STEP 18 — Multi-Turn Conversation Prompt

In [ ]:
def conversation_prompt():
    history = "\n".join(
        f"{m['role'].upper()}: {m['message']}" for m in conversation
    )
    return f'''You are TechKart support.
Conversation history:
{history}

Respond to the latest customer message using the conversation context.
Do not invent information.'''

print(conversation_prompt())

# STEP 19 — Mini Support Dashboard

In [ ]:
display(pd.DataFrame(tickets) if tickets else pd.DataFrame())

# STEP 20 — End-to-End Demonstration

In [ ]:
def run_complete_support_demo(message, customer_name="Customer"):
    result = support_agent(message, customer_name)

    print("="*70)
    print("CUSTOMER:", message)
    print("\nANALYSIS:")
    print(json.dumps(result["analysis"], indent=2))
    print("\nORDER:")
    print(json.dumps(result["order_info"], indent=2))
    print("\nRETRIEVED POLICY:", result["policy"]["policy"])
    print("\nDECISION:")
    print(json.dumps(result["decision"], indent=2))

    response = result["decision"].get("customer_response", "")
    evaluation = evaluate_response(message, response, result)

    print("\nEVALUATION:")
    print(json.dumps(evaluation, indent=2))

    human = needs_human_review(result["analysis"], result["order_info"])
    print("\nHUMAN REVIEW REQUIRED:", human)

    if human:
        print(json.dumps(
            escalate_to_human("High-risk or high-priority case", result["ticket"]),
            indent=2
        ))
    print("="*70)

run_complete_support_demo(
    "My laptop hasn't arrived yet. Expected delivery was yesterday. Order ORD1001.",
    "Rahul"
)

# 🧪 STEP 21 — Lets Check Project output in Localhost through CMD

Assign teams:

1. Delivery Agent
2. Refund Agent
3. Damaged Product Agent
4. Wrong Product Agent
5. Technical Support Agent
6. Angry Customer / Escalation Agent

### Every team must demonstrate
1. AI classification
2. Order ID extraction
3. Tool call
4. RAG retrieval
5. Agent decision
6. Final response
7. Response evaluator
8. Human escalation when appropriate.

# 🚀 STEP 23 — Production Architecture

```text
Web / Mobile / WhatsApp
          ↓
       FastAPI
          ↓
   Customer Support Agent
      ↙     ↓      ↘
    LLM     RAG    Tools
            ↓       ↓
         Vector DB  CRM / Orders / ERP
              ↓
          Guardrails
              ↓
           Evaluator
              ↓
        Human Escalation
              ↓
          Analytics
```

Possible production stack:
- Python + FastAPI
- Gemini / other enterprise LLM
- PostgreSQL/MySQL
- FAISS/Chroma/Pinecone/Weaviate
- React/Next.js
- Docker/cloud deployment
- CRM/helpdesk/WhatsApp integrations

# 🏆 Final Learning Outcome

Students have built a simplified **AI-powered business workflow**, not just a chatbot:

**LLM + Prompt Engineering + Structured Output + RAG + Tools + Memory + Agentic Decision Making + Evaluation + Human-in-the-Loop**

The same architecture can be adapted to banking, healthcare support, education, travel, e-commerce, HR, and enterprise helpdesk systems.